## Import

In [57]:
from pdf2image import convert_from_path
import openai 
from openai import OpenAI
import json
import re
from dotenv import load_dotenv
import os
import easyocr
from typing import List, Tuple , Union
import numpy as np
import logging
import time
from concurrent.futures import ThreadPoolExecutor
import cv2

#### API key

In [58]:
load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print("API Key:", api_key)

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

API Key: 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


In [59]:
load_dotenv(override=True)
openai.api_key = os.getenv("OPENAI_API_KEY") 
assert openai.api_key, "Cannot find OPENAI_API_KEY in .env"

In [60]:
# POPPLER_PATH = r"C:/Users/Ned/Desktop/Poppler/poppler-24.08.0/Library/bin"

#### Import file to use OCR
- Change file -> img

In [61]:
folder_path = "./"
pdf_pattern = re.compile(r".+\.pdf$", re.IGNORECASE)

pdf_files = [f for f in os.listdir(folder_path) if pdf_pattern.match(f)]

if len(pdf_files) == 0:
    raise FileNotFoundError("Not found PDF file")
else:
    pdf_file = pdf_files[0]
    print(f"Using: {pdf_file}")

    images = convert_from_path(pdf_file, dpi=200)
    print(f"PDF conversion completed {len(images)} pages")

Using: 168001084 (1).pdf
PDF conversion completed 2 pages


#### Use EasyOCR

In [62]:
# === Setup Logging ===
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s: %(message)s',
    datefmt='%H:%M:%S'
)

# === Create EasyOCR Reader ===
reader = easyocr.Reader(['th', 'en'], gpu=False)

# === Define output file ===
out_path = "output_corrected.txt"

# === OCR Phase ===
def ocr_page(pg_img_tuple: Tuple[int, np.ndarray]) -> Tuple[int, List[str]]:
    pg, img = pg_img_tuple
    logging.info(f"📸 OCR Processing Page {pg}")
    results = reader.readtext(np.array(img))
    texts = [raw_text for _, raw_text, _ in results]
    logging.info(f"🔍 OCR done Page {pg}, found {len(texts)} items")
    return (pg, texts)

# === Correction Phase using AI ===
def correct_text_with_ai(text: str) -> str:
    prompt = (
        "คุณเป็นผู้ช่วยที่ช่วยแก้ไขข้อความ OCR ภาษาไทยให้ถูกต้อง "
        "(รวมถึงคำผิดจาก OCR และเว้นวรรคผิด) "
        "โดยไม่เปลี่ยนแปลงความหมายของข้อความ ตอบกลับเฉพาะข้อความที่แก้ไขแล้วเท่านั้น:\n\n"
        f"{text}"
    )

    start_time = time.time()
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    elapsed = time.time() - start_time
    logging.info(f"⏳ GPT responded in {elapsed:.2f} seconds")

    return response.choices[0].message.content.strip()

# Batch Correction
MAX_LINES = 30

def batch_correct_with_ai(texts: List[str]) -> List[str]:
    def correct_chunk(chunk):
        logging.info(f"🧠 Sending {len(chunk)} lines to GPT")
        return correct_text_with_ai("\n".join(chunk)).split("\n")

    all_fixed = []
    chunks = [texts[i:i+MAX_LINES] for i in range(0, len(texts), MAX_LINES)]

    with ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(correct_chunk, chunks))

    for corrected_lines in results:
        all_fixed.extend(corrected_lines)

    return all_fixed


# === Function to process each page in a thread === (( OCR + GPT ))
def process_page(pg_img_tuple: Tuple[int, np.ndarray]) -> str:
    pg, img = pg_img_tuple
    start_time = time.time()
    logging.info(f"🟡 Processing Page {pg}")

    text_output = f"\n📄 Page {pg}\n"

    results = reader.readtext(np.array(img))
    texts = [raw_text for _, raw_text, _ in results]
    logging.info(f"🔍 Found {len(texts)} text items on Page {pg}")

    fixed_texts = batch_correct_with_ai(texts)

    for raw_text, fixed_text in zip(texts, fixed_texts):
        text_output += f"❌ {raw_text}\n"
        text_output += f"✅ {fixed_text}\n\n"

    elapsed = time.time() - start_time
    logging.info(f"✅ Finished Page {pg} in {elapsed:.2f} seconds")
    return text_output

# === Run OCR + Correction in parallel using ThreadPoolExecutor ===
logging.info("🚀 Starting OCR + GPT with separate phases")
start_time_all = time.time()

with ThreadPoolExecutor(max_workers=2) as executor:  # <--- ThreadPoolExecutor 2 Cores
    outputs = list(executor.map(process_page, enumerate(images, start=1)))

# === Save output ===
with open(out_path, 'w', encoding='utf-8') as fout:
    fout.writelines(outputs)

total_time = time.time() - start_time_all
logging.info(f"✅ All done. Output saved to {out_path}")
logging.info(f"⏱️ Total processing time: {total_time:.2f} seconds")

[19:39:20] WARNING: Using CPU. Note: This module is much faster with a GPU.
[19:39:23] INFO: 🚀 Starting OCR + GPT with separate phases
[19:39:23] INFO: 🟡 Processing Page 1
[19:39:23] INFO: 🟡 Processing Page 2
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
[19:40:29] INFO: 🔍 Found 88 text items on Page 2
[19:40:29] INFO: 🧠 Sending 30 lines to GPT
[19:40:29] INFO: 🧠 Sending 30 lines to GPT
[19:40:29] INFO: 🧠 Sending 28 lines to GPT
[19:40:30] INFO: 🔍 Found 89 text items on Page 1
[19:40:30] INFO: 🧠 Sending 30 lines to GPT
[19:40:30] INFO: 🧠 Sending 30 lines to GPT
[19:40:30] INFO: 🧠 Sending 29 lines to GPT
[19:40:30] INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
[19:40:30] INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
[19:40:3

#### Extract key field by use AI

In [63]:
# === Extract key field ===
def extract_fields(text: str) -> dict:
    system_prompt = (
    "You are an AI assistant that extracts key information from Thai government financial documents using OCR text. "
    "Please extract the following fields and return only in JSON format:\n\n"
    "- bill_number: เลขที่ใบขอซื้อ เช่น 10778\n"
    "- bill_type: ประเภทเอกสาร เช่น รายงานขออนุมัติจัดจ้าง\n"
    "- supplier_name: หน่วยงานหรือชื่อผู้ขาย หรือ ชื่อซัพพลายเออร์\n"
    "- amount: ยอดรวมสุทธิที่อยู่ใกล้คำว่า 'รวมทั้งสิ้น' หรือ 'ยอดรวม' หรือ 'รวมจำนวนเงิน' หรือ 'รวม' หรือคำที่สามารถบอกว่าตัวเลขนั้นคือยอดรวมทั้งหมด\n"
    "- payment_date: วันที่ใด ๆ ในเอกสาร \n"
    "- signature: 'อนุมัติ' หรือ 'ผู้เบิก' และนำชื่อจริงนามสกุลเท่านั้นออกมา ตัดคำนำหน้า หรือ  ตำแหน่ง ออก)\n\n"

    "If any field is not found, use null. Respond in JSON format only without any explanation."
    )
    user_prompt = f"OCR Text in Thai:\n{text}\n\nPlease return the result in JSON only."

    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    content = response.choices[0].message.content.strip()
    print("📥 AI response:\n", content)

    try:
        return json.loads(content)
    except json.JSONDecodeError as e:
        print("❌ JSON parsing failed:", e)
        return {"error": "Invalid JSON", "raw": content}


# === Loading OCR+Correction processed files ===
with open("output_corrected.txt", "r", encoding="utf-8") as f:
    ocr_text = f.read()

# Separate page (ตาม marker 📄 Page X)
pages = re.split(r"\n📄 Page \d+\n", ocr_text)
pages = [p.strip() for p in pages if p.strip()]

# === Extract key field each page ===
all_extracted = []
for i, page_text in enumerate(pages, start=1):
    print(f"\n🔍 Extracting from Page {i}...")
    extracted = extract_fields(page_text)
    extracted['page'] = i
    all_extracted.append(extracted)

# === Display result ===
print("\n📑 Extracted All Receipts:")
for receipt in all_extracted:
    print(json.dumps(receipt, ensure_ascii=False, indent=2))


🔍 Extracting from Page 1...


[19:40:48] INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


📥 AI response:
 {
    "bill_number": "168001084",
    "bill_type": "ใบสำคัญการตั้งหนี้",
    "supplier_name": "แวนชาส ฺทราเวล โดย นางสาวอรทัย",
    "amount": "2,200.00",
    "payment_date": "มี.ค. 2568",
    "signature": "นางสาวอรทัย"
}

🔍 Extracting from Page 2...


[19:40:51] INFO: HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


📥 AI response:
 {
    "bill_number": "168001174",
    "bill_type": "ใบสำคัญการตั้งหนี้",
    "supplier_name": "บริษัท สหายไฟฟ้า เอ็นจิเนียริ่ง จำกัด",
    "amount": "73,085.00",
    "payment_date": "21 มี.ค. 2568",
    "signature": "คณบดีวิทยาลัยศิลปะ เซนต์ และเทคโนโลยี"
}

📑 Extracted All Receipts:
{
  "bill_number": "168001084",
  "bill_type": "ใบสำคัญการตั้งหนี้",
  "supplier_name": "แวนชาส ฺทราเวล โดย นางสาวอรทัย",
  "amount": "2,200.00",
  "payment_date": "มี.ค. 2568",
  "signature": "นางสาวอรทัย",
  "page": 1
}
{
  "bill_number": "168001174",
  "bill_type": "ใบสำคัญการตั้งหนี้",
  "supplier_name": "บริษัท สหายไฟฟ้า เอ็นจิเนียริ่ง จำกัด",
  "amount": "73,085.00",
  "payment_date": "21 มี.ค. 2568",
  "signature": "คณบดีวิทยาลัยศิลปะ เซนต์ และเทคโนโลยี",
  "page": 2
}
